# Implementación

## Paso 0. Importación de bibliotecas

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import flopy
import xmf6

## Paso 1. Parámetros físicos.

In [ ]:
phys = dict(
    specific_discharge = 0.2,  # Specific discharge ($m s^{-1}$)
    hydraulic_conductivity = 1.0,  # Hydraulic conductivity ($m s^{-1}$)
    source_concentration = contenido_44,  # Source concentration (unitless)
    porosity = 0.5,  # Porosity of mobile domain (unitless)
    initial_concentration = 0.0,  # Initial concentration (unitless)
)
xmf6.nice_print(phys, "Parámetros físicos")

## Paso 2. Simulación del flujo: GWF

In [ ]:
# Parámetros de la simulación (flopy.mf6.MFSimulation)
init_f = {
    'sim_name' : "flow",
    'exe_name' : r"C:\Users\luiggi\Documents\GitSites\mf6_tutorial\mf6\windows\mf6",
#    'exe_name' : "../../mf6/macosarm/mf6",
    'sim_ws' : "output_flow_trans_1D"
}

# Parámetros para el tiempo (flopy.mf6.ModflowTdis)
tdis_f = {
    'units': "days", #"seconds",
    'nper' : 1,
    'perioddata': [(dt, 1, 1.0)] #[(120.0, 240, 1.0)] (duración del periodo,num.de.pasos.de.tiempo,multiplicador)
}

# Parámetros para la solución numérica (flopy.mf6.ModflowIms)
# OJO: definiremos el modelo de solución más adelante
ims_f = {} 

# Parámetros para el modelo de flujo (flopy.mf6.ModflowGwf)
gwf = { 
    'modelname': init_f["sim_name"],
    'save_flows': True
}

# Parámetros para la discretización espacial (flopy.mf6.ModflowGwfdis)
# --- Diccionario dis ---
#ncolt=10 # Al cambiar ncolt, conc_i.m3d debe tener ncolt columnas
#conc_i_m3d = {i: 1 for i in range(ncolt)}
#print(conc_i_m3d)

dis = {
    'length_units' : "meters",
    'nlay': 1,  
    'nrow': 1,  
    'ncol': ncolt, 
    'delr': dely,
    'delc': delx,
    'top' : delz,
    'botm': 0.0 
}

# Parámetros para las condiciones iniciales (flopy.mf6.ModflowGwfic)
ic = {
    'strt': 1.0 #'strt':conc_inicial 
}

# Parámetros para las condiciones de frontera (flopy.mf6.ModflowGwfchd)
chd = {
    'stress_period_data': [[(0, 0, dis['ncol'] - 1), 1.0]],     
}

# Parámetros para las propiedades de flujo (flopy.mf6.ModflowGwfnpf)
npf = {
    'save_specific_discharge': True,
    'save_saturation': True,
    'icelltype':  0,
    'k': phys["hydraulic_conductivity"] # Hydraulic conductivity ($cm s^{-1}$) 
}

# Parámetros para las propiedades de los pozos (flopy.mf6.ModflowGwfwel)
q = phys["specific_discharge"] * dis['delc'] * dis['delr'] * dis['top']
well = {
    'stress_period_data': [[(0, 0, 0), q, phys["source_concentration"],]],
    'pname': "WEL-1",
    'auxiliary' : ["CONCENTRATION"],
#    'save_flows': True
}

# Parámetros para almacenar y mostrar la salida de la simulación (flopy.mf6.ModflowGwfoc)
oc = {
    'budget_filerecord': f"{init_f['sim_name']}.bud",
    'head_filerecord': f"{init_f['sim_name']}.hds",
    'saverecord': [("HEAD", "ALL"), ("BUDGET", "ALL")],
#    'printrecord': [("HEAD", "ALL")]
}

# --- Inicialización de la simulación ---
o_sim = xmf6.common.init_sim(silent = True, init = init_f, tdis = tdis_f, ims = ims_f)
#o_sim_f = xmf6.common.init_sim(silent = True, init = init_f, tdis = tdis_f, ims = ims_f)
o_gwf, packages = xmf6.gwf.set_packages(o_sim, silent = True,
                                        gwf = gwf, 
                                        dis = dis, ic = ic, chd = chd, npf = npf, oc = oc, well = well)
# --- Escritura de archivos ---
o_sim.write_simulation(silent = True)

# --- Ejecución de la simulación ---
o_sim.run_simulation(silent = True)
#o_sim.run_simulation()